In [ ]:
import numpy as np
import ot
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import time
from datetime import datetime

In [ ]:
dimensions = [5, 10, 15] # dimension
N = 10000 # support size of base measures
sample_sizes = np.arange(100,1001,100)
T = 5 # number of iterations per sample size

In [ ]:
# Setting A in paper

def k_star(x):
  y = x.copy()
  if np.random.rand() < 0.5:
    y[0] += 1
  else:
    y[0] -= 1
  return y

np.random.seed(1234)
mu_by_dimension = {}
nu_by_dimension = {}
for d in dimensions:
  mu_by_dimension[d] = np.concatenate((np.zeros((N,1)), np.random.rand(N,d-1)), axis=1)
  nu_by_dimension[d] = np.concatenate((np.zeros((N,1)), np.random.rand(N,d-1)), axis=1)
  nu_by_dimension[d] = np.apply_along_axis(k_star, 1, nu_by_dimension[d])

In [ ]:
max_iters = 500000 # increase if this is saturated

NN_E1_errors = {}
NN_L1_errors = {}
NN_opt_gaps = {}
NN_feas_gaps = {}
for d in dimensions:
  print(f'd:{d}')
  mu = mu_by_dimension[d]
  nu = nu_by_dimension[d]

  print('base dist computation')
  C = ot.dist(mu, nu, metric='euclidean')
  pi_N = ot.emd(np.ones(N)/N, np.ones(N)/N, C, numItermax=max_iters)
  sigma_N = pi_N.argmax(axis=1)
  opt_destinations = nu[sigma_N]
  print('done with base dist computation')

  for n in sample_sizes:
    print(f'n:{n}')
    NN_E1_errors[(d,n)] = []
    NN_opt_gaps[(d,n)] = []
    NN_feas_gaps[(d,n)] = []
    NN_L1_errors[(d,n)] = []
    for t in range(T):
      indices_mu = np.random.choice(N, n, replace=True)
      indices_nu = np.random.choice(N, n, replace=True)
      mu_n = mu[indices_mu,:]
      nu_n = nu[indices_nu,:]

      # compute NN estimator
      C = ot.dist(mu_n, nu_n, metric='euclidean')
      pi_n = ot.emd(np.ones(n)/n, np.ones(n)/n, C, numItermax=max_iters)
      sigma_n = pi_n.argmax(axis=1)
      tree = cKDTree(mu_n)
      _, nn_indices = tree.query(mu)
      destinations = nu_n[sigma_n[nn_indices]]

      NN_L1_errors[(d,n)].append(np.linalg.norm(destinations - opt_destinations, axis=1).mean())

      L1_transport_cost = np.linalg.norm(destinations - mu, axis=1).mean()
      opt_gap = max(L1_transport_cost - 1,0)
      C = ot.dist(destinations, nu, metric='euclidean')
      feas_gap = ot.emd2(np.ones(N)/N, np.ones(N)/N, C)
      NN_opt_gaps[(d,n)].append(opt_gap)
      NN_feas_gaps[(d,n)].append(feas_gap)
      NN_E1_errors[(d,n)].append(opt_gap + feas_gap)

In [ ]:
rounding_E1_errors = {}
rounding_opt_gaps = {}
rounding_feas_gaps = {}
for d in dimensions:
  print(f'd:{d}')
  mu = np.concatenate((np.zeros((N,1)), np.random.rand(N,d-1)), axis=1)
  nu = np.concatenate((np.zeros((N,1)), np.random.rand(N,d-1)), axis=1)
  nu = np.apply_along_axis(k_star, 1, nu)

  for n in sample_sizes:
    print(f'n:{n}')
    rounding_E1_errors[(d,n)] = []
    rounding_opt_gaps[(d,n)] = []
    rounding_feas_gaps[(d,n)] = []
    for t in range(T):
      indices_mu = np.random.choice(N, n, replace=True)
      indices_nu = np.random.choice(N, n, replace=True)
      mu_n = mu[indices_mu,:]
      nu_n = nu[indices_nu,:]

      # compute rounding estimator
      delta = n**(-1/(d+2))
      mu_n_rounded = delta * np.round(mu_n / delta)
      C = ot.dist(mu_n_rounded, nu_n, metric='euclidean')
      pi_n = ot.emd(np.ones(n)/n, np.ones(n)/n, C)

      mu_rounded = delta * np.round(mu / delta)
      tree = cKDTree(mu_n_rounded)
      _, nn_indices = tree.query(mu_rounded)
      kernel_dists_raw = pi_n[nn_indices,:] # N x n
      row_sums = kernel_dists_raw.sum(axis=1, keepdims=True)
      kernel_dists = kernel_dists_raw / row_sums

      L1_transport_cost = 0
      for k in range(N):
        for l in range(n):
          L1_transport_cost += np.linalg.norm(mu[k,:] - nu_n[l,:]) * kernel_dists[k,l]
      L1_transport_cost /= N

      destination_weights = kernel_dists.mean(axis=0)

      opt_gap = max(L1_transport_cost - 1,0)
      C = ot.dist(nu_n, nu, metric='euclidean')
      feas_gap = ot.emd2(destination_weights, np.ones(N)/N, C)
      rounding_opt_gaps[(d,n)].append(opt_gap)
      rounding_feas_gaps[(d,n)].append(feas_gap)
      rounding_E1_errors[(d,n)].append(opt_gap + feas_gap)

In [ ]:
x = sample_sizes
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for k, ax in enumerate(axes):
    for i, d in enumerate(dimensions):
        color = colors[i % len(colors)]

        # Select y and y2 depending on subplot
        if k == 0:
            y = [np.mean(rounding_E1_errors[(d, n)]) for n in sample_sizes]
            y2 = [np.mean(NN_E1_errors[(d, n)]) for n in sample_sizes]
        elif k == 1:
            y = [np.mean(rounding_opt_gaps[(d, n)]) for n in sample_sizes]
            y2 = [np.mean(NN_opt_gaps[(d, n)]) for n in sample_sizes]
        else:  # k == 2
            y = [np.mean(rounding_feas_gaps[(d, n)]) for n in sample_sizes]
            y2 = [np.mean(NN_feas_gaps[(d, n)]) for n in sample_sizes]

        # Plot NN and Rounding
        ax.plot(x, y2, label=f'd={d},{"  " if d < 10 else ""} NN', color=color)
        ax.plot(x, y, '--', label=f'           rounding', color=color)

    # Log-log scale
    ax.set_xscale('log')
    ax.set_yscale('log')

    # Labels and title
    ax.set_xlabel('n (log scale)', fontsize=14)
    if k == 0:
        ax.set_ylabel('Error (log scale)', fontsize=14)
        ax.set_title('ℰ₁: NN vs. Rounding', fontsize=16)
    elif k == 1:
        ax.set_title('Optimality Gap: NN vs. Rounding', fontsize=16)
    else:  # k == 2
        ax.set_title('Feasibility Gap: NN vs. Rounding', fontsize=16)

    # Tick parameters
    ax.tick_params(axis='both', which='major', labelsize=12)

    # Grid
    ax.grid(True, which="both", ls="--", linewidth=0.5)

    # Legend with larger font
    ax.legend(loc='best', fontsize=12)

plt.tight_layout()
plt.show()